In [117]:
#Carregando dados e Bibliotecas

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [118]:
print('=' *60)
print("--- Tarefa 1.7 - segmentação e recomendação ---")
print('=' * 60)
print(" Passo 1: Carregando Dados e Criando Feature do Cliente")

--- Tarefa 1.7 - segmentação e recomendação ---
 Passo 1: Carregando Dados e Criando Feature do Cliente


In [119]:
print('\n' + '=' * 60)
print('passo 1: carregando dados e criando features do cliente')
print('=' * 60)


passo 1: carregando dados e criando features do cliente


In [120]:
print('\n Carregando os arquivos necessarios')

df_orders = pd.read_csv(r'C:\GameStoreBrasil\output\orders_cleaned.csv', parse_dates=['order_date'] )
df_games = pd.read_csv(r'C:\GameStoreBrasil\data\games.csv', parse_dates=['release_date'])
df_members = pd.read_csv(r'C:\GameStoreBrasil\output\members_cleaned.csv', parse_dates=['join_date'])


 Carregando os arquivos necessarios


In [121]:
print(f'orders_cleaned.csv tem {len(df_orders)} linhas')
print(f'members_cleaned.csv tem {len(df_members)} linhas')

orders_cleaned.csv tem 5000 linhas
members_cleaned.csv tem 300 linhas


In [122]:
#Limpar df orders 
print("filtrando valores negativos")
df_orders = df_orders[(df_orders['quantity']>0) & (df_orders['unit_price'] > 0)]



filtrando valores negativos


In [123]:
#Criando revenue

df_orders['revenue'] = df_orders['quantity'] * df_orders['unit_price']

In [124]:
#Feature, agrupar clientes por member id, features de total de compras e ticket medio 

features = df_orders.groupby('member_id').agg(
    total_purchase = ('order_id', 'count'),
    avg_purchase_value = ('revenue', 'mean')
).reset_index()
x1 = df_orders.groupby('member_id')['quantity']
print('\n x1 aqui')
print(x1.head(3))


 x1 aqui
0       2
1       1
2       3
3       2
4       2
       ..
4671    1
4680    3
4772    3
4856    1
4988    1
Name: quantity, Length: 950, dtype: int64


In [125]:
print(f' - Clientes com pelo menos 1 compra: {len(features)}')
print(features.head(20))

 - Clientes com pelo menos 1 compra: 350
    member_id  total_purchase  avg_purchase_value
0           1              10          243.788000
1           2              13          386.800769
2           3              12          362.271667
3           4              12          409.600833
4           5              20          330.159000
5           6              15          310.742667
6           7              17          444.294118
7           8              11          409.331818
8           9              20          435.265000
9          10              13          389.057692
10         11              12          409.828333
11         12              18          461.630556
12         13              17          390.312353
13         14              12          397.041667
14         15              13          500.234615
15         16              19          376.344737
16         17              16          431.646250
17         18              11          364.730000
18       

In [126]:
#5 Tratamento de clientes sem compra

features = df_members[['member_id']].merge(features, on ='member_id', how = 'left')

#preencher os vazios (NaN) com 0 para quem nunca comprou 

features['total_purchase'] = features['total_purchase'].fillna('0').astype(int)
features['avg_purchase_value'] = features['avg_purchase_value'].fillna(0.0)

In [127]:
print(f' - Total de clientes no dataset final (incluindo quem nunca comprou) {len(features)}')

print(f'Clientes com com zero compras: {(features['total_purchase'] == 0).sum()}')

print("\n ---- Primeiras 10 linhas do DataFrame de Features ----")
print(features.head(10))

 - Total de clientes no dataset final (incluindo quem nunca comprou) 300
Clientes com com zero compras: 0

 ---- Primeiras 10 linhas do DataFrame de Features ----
   member_id  total_purchase  avg_purchase_value
0          1              10          243.788000
1          2              13          386.800769
2          3              12          362.271667
3          4              12          409.600833
4          5              20          330.159000
5          6              15          310.742667
6          7              17          444.294118
7          8              11          409.331818
8          9              20          435.265000
9         10              13          389.057692


In [128]:
#07 Estatisticas descritivas das novas features 

print('\n ---- Estatisticas descritivas das features ----')
print(features[['total_purchase','avg_purchase_value']].describe())


 ---- Estatisticas descritivas das features ----
       total_purchase  avg_purchase_value
count      300.000000          300.000000
mean        15.916667          372.681239
std          3.930379           66.247065
min          6.000000          168.784118
25%         13.000000          328.018614
50%         16.000000          370.389215
75%         18.000000          414.318374
max         28.000000          578.477000


In [129]:
print('\n' + '=' * 60)

print('passo 2, Aplicando o algoritmo  K-Means')

print("=" * 60)


passo 2, Aplicando o algoritmo  K-Means


In [130]:
#1 Selecionar apenas as colunas numéricas que serão usadas pelo modelo (A Matriz X)
# O K-Means so entende números, então isolamos as features que criamos no passo 1

x = features[['total_purchase', 'avg_purchase_value']]


In [131]:
#2 Instanciar o Modelo K-Means
#n clusters = 3: Queremos 3 grupos(Conforme o Enunciado)

In [132]:
print('\n Treinando o modelo K-Means com 3 clusters')
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels = kmeans.fit_predict(x)

features['cluster_label'] = labels + 1
print(features.head(20))


 Treinando o modelo K-Means com 3 clusters
    member_id  total_purchase  avg_purchase_value  cluster_label
0           1              10          243.788000              2
1           2              13          386.800769              1
2           3              12          362.271667              1
3           4              12          409.600833              1
4           5              20          330.159000              2
5           6              15          310.742667              2
6           7              17          444.294118              3
7           8              11          409.331818              1
8           9              20          435.265000              3
9          10              13          389.057692              1
10         11              12          409.828333              1
11         12              18          461.630556              3
12         13              17          390.312353              1
13         14              12          397.041

In [133]:
print(f'Modelo treinado com sucesso')
print(f'Rotulos Atribuidos: {sorted(features['cluster_label'].unique())}')

Modelo treinado com sucesso
Rotulos Atribuidos: [np.int32(1), np.int32(2), np.int32(3)]


In [134]:
#Calcular os centreoids, a Média real de cada cluster 
#Isso nos diz o perfil de cada grupo, exemplo, o cluster 1 são os que compram muito

centroids = (
    features
    .groupby('cluster_label')
    [['total_purchase', 'avg_purchase_value']]
    .mean()
    .round(2)
)
print(features)

     member_id  total_purchase  avg_purchase_value  cluster_label
0            1              10          243.788000              2
1            2              13          386.800769              1
2            3              12          362.271667              1
3            4              12          409.600833              1
4            5              20          330.159000              2
..         ...             ...                 ...            ...
295        296              20          356.774500              1
296        297              10          469.419000              3
297        298              19          353.557368              1
298        299              17          428.293529              3
299        300              17          324.454706              2

[300 rows x 4 columns]


In [135]:
print("\n Centroides dos Clusters (Perfil de cada grupo)")
print(centroids)

#Verificar o tamanho de cada cluster(Quantos clientes cairam em cada grupo?)

cluster_sizes = features['cluster_label'].value_counts().sort_index()


 Centroides dos Clusters (Perfil de cada grupo)
               total_purchase  avg_purchase_value
cluster_label                                    
1                       16.36              380.85
2                       15.81              302.01
3                       15.12              464.53


In [136]:
print('\n --- tamanho dos clusters (Distribuição de clientes) ---')
print(cluster_sizes.head(10))
for cluster, size in cluster_sizes.items():
    print(f' - Cluster {cluster} : {size} Clientes')

#Passo 3: Afinidade de Produtos por cluster


 --- tamanho dos clusters (Distribuição de clientes) ---
cluster_label
1    137
2     99
3     64
Name: count, dtype: int64
 - Cluster 1 : 137 Clientes
 - Cluster 2 : 99 Clientes
 - Cluster 3 : 64 Clientes


In [137]:
print('\n' + '=' * 60)
print("Passo 3: Calculando A Afinidade de Produtos por Cluster")
print("=" * 60)

#1. Trazer a informação do cluster para o nivel de transação


Passo 3: Calculando A Afinidade de Produtos por Cluster


In [138]:
df_orders_with_cluster = df_orders.merge(
    features[['member_id', 'cluster_label']],
    on = 'member_id',
    how = 'left'
)
print(df_orders_with_cluster.head(10))
print(f' - transacoes enriquecidas com cluster: {len(df_orders_with_cluster)}')

   order_id  member_id game_id  quantity  unit_price          order_date  \
0         1        106   G0067         2      207.06 2025-02-01 09:11:00   
1         2        233   G0006         1       65.40 2024-11-27 15:02:00   
2         3        267   G0036         3       99.02 2024-09-26 14:37:00   
3         4        292   G0023         2       52.23 2025-01-22 12:40:00   
4         5         40   G0067         2      207.06 2025-01-20 12:46:00   
5         6        265   G0052         2      212.87 2024-12-19 16:57:00   
6         7        247   G0078         1      244.12 2025-05-28 09:29:00   
7         8         85   G0030         1      212.39 2025-06-10 15:49:00   
8         9        109   G0020         1       74.45 2024-06-10 10:42:00   
9        10         61   G0030         2      212.39 2024-12-04 09:16:00   

  promotion_id  revenue  cluster_label  
0            0   414.12            1.0  
1            0    65.40            2.0  
2            0   297.06            3.0  

In [139]:
affinity = df_orders_with_cluster.groupby(['cluster_label', 'game_id']).size().reset_index(name = 'purchase_count')
print(affinity.head(300))
print(affinity.loc[affinity['purchase_count'].max()])

     cluster_label game_id  purchase_count
0              1.0   G0001              30
1              1.0   G0002              27
2              1.0   G0003              33
3              1.0   G0004              22
4              1.0   G0005              24
..             ...     ...             ...
282            3.0   G9033               1
283            3.0   G9036               1
284            3.0   G9038               1
285            3.0   G9041               1
286            3.0   G9045               1

[287 rows x 3 columns]
cluster_label       1.0
game_id           G0042
purchase_count       30
Name: 41, dtype: object


In [140]:
affinity = affinity.sort_values(
    by = ['cluster_label', 'purchase_count'],
    ascending= [True, False]
).reset_index(drop = True)

In [141]:
print("\n Top 3 jogos mais comprados por cluester ( afinidade ")

for cluster in [1,2,3]:
    print(f'\n Cluster {cluster}')
    #Filtrar o cluster atual e pega as 3 primeiras linhas que ja estao ordenadas 
    top_games = affinity[affinity['cluster_label'] == cluster]
    for _, row in top_games.iterrows():
        print(f' -{row['game_id']} : {row['purchase_count']} transacoes')

print("\n" + "=" * 60)
print("PASSO 4: GERANDO RECOMENDACOES (REGRA DE NETFLIX)")
print("\n" + "=" * 60)


 Top 3 jogos mais comprados por cluester ( afinidade 

 Cluster 1
 -G0055 : 41 transacoes
 -G0036 : 40 transacoes
 -G0074 : 40 transacoes
 -G0006 : 38 transacoes
 -G0037 : 38 transacoes
 -G0056 : 38 transacoes
 -G0061 : 38 transacoes
 -G0051 : 37 transacoes
 -G0078 : 37 transacoes
 -G0020 : 35 transacoes
 -G0050 : 35 transacoes
 -G0076 : 35 transacoes
 -G0012 : 34 transacoes
 -G0015 : 34 transacoes
 -G0022 : 34 transacoes
 -G0034 : 34 transacoes
 -G0049 : 34 transacoes
 -G0052 : 34 transacoes
 -G0059 : 34 transacoes
 -G0069 : 34 transacoes
 -G0003 : 33 transacoes
 -G0047 : 33 transacoes
 -G0040 : 31 transacoes
 -G0043 : 31 transacoes
 -G0067 : 31 transacoes
 -G0001 : 30 transacoes
 -G0011 : 30 transacoes
 -G0024 : 30 transacoes
 -G0042 : 30 transacoes
 -G0063 : 30 transacoes
 -G0025 : 29 transacoes
 -G0039 : 29 transacoes
 -G0058 : 29 transacoes
 -G0068 : 29 transacoes
 -G0026 : 28 transacoes
 -G0029 : 28 transacoes
 -G0046 : 28 transacoes
 -G0048 : 28 transacoes
 -G0002 : 27 transaco

In [142]:
#Criar um dicionario com a lista de jogos ordenados por popularidade para cada cluster 

top_candidatos = affinity.groupby('cluster_label').head(20)
cluster_to_games = top_candidatos.groupby('cluster_label')['game_id'].apply(list).to_dict()


In [143]:
#2 Criar um dicionario com o historico de compras de cada cliente

client_history = df_orders.groupby('member_id')['game_id'].apply(set).to_dict()
print(client_history)

print('\n Gerando recomendações personalizadas para 300 clientes')
lista_recomendacoes = []

{1: {'G0009', 'G0074', 'G0007', 'G0010', 'G0057', 'G0020', 'G0040', 'G0035', 'G0006', 'G0037'}, 2: {'G0056', 'G0052', 'G0067', 'G0061', 'G0074', 'G0059', 'G0034', 'G0015', 'G0055', 'G0051', 'G0058', 'G0003', 'G0078'}, 3: {'G0050', 'G0054', 'G0059', 'G0068', 'G0062', 'G0042', 'G0051', 'G0012', 'G0013', 'G0001', 'G0043'}, 4: {'G0063', 'G0022', 'G0039', 'G0049', 'G0069', 'G0074', 'G0023', 'G0047', 'G0013', 'G0037'}, 5: {'G0059', 'G0007', 'G0077', 'G0019', 'G0051', 'G0063', 'G0032', 'G0021', 'G0002', 'G0056', 'G0030', 'G0075', 'G0069', 'G0047', 'G0038', 'G0027', 'G0023', 'G0066', 'G0036', 'G0031'}, 6: {'G0004', 'G0030', 'G0024', 'G0074', 'G0007', 'G0037', 'G0023', 'G0042', 'G0021', 'G0058', 'G0064', 'G0043', 'G0035', 'G0078'}, 7: {'G0009', 'G0018', 'G0014', 'G0022', 'G0063', 'G0075', 'G0061', 'G0016', 'G0044', 'G0027', 'G0068', 'G0015', 'G0021', 'G0064', 'G0002', 'G0035', 'G0078'}, 8: {'G0022', 'G0050', 'G0061', 'G0037', 'G0019', 'G0020', 'G0025', 'G0053', 'G0036', 'G0035'}, 9: {'G0009', '

In [144]:
for _, cliente in features.iterrows():
    mid = cliente['member_id']
    cluster = cliente['cluster_label']
   
    # Pega a lista de jogos populares do cluster do cliente
    jogos_populares = cluster_to_games.get(cluster, [])
   
    # Pega o histórico do cliente (se ele nunca comprou, retorna um set vazio)
    jogos_comprados = client_history.get(mid, set())
   
    # O FILTRO MÁGICO: Queremos jogos populares, MAS que o cliente NÃO comprou
    jogos_sugeridos = [jogo for jogo in jogos_populares if jogo not in jogos_comprados]
   
    # Pega apenas os 3 primeiros da lista filtrada
    top3 = jogos_sugeridos[:3]
   
    # Tratamento de segurança: caso o cliente já tenha comprado todos os top 20 (muito raro)
    while len(top3) < 3:
        top3.append("N/A")
       
    # Adiciona o resultado na nossa lista
    lista_recomendacoes.append({
        "member_id": mid,
        "cluster_label": cluster,
        "recommended_game_1": top3[0],
        "recommended_game_2": top3[1],
        "recommended_game_3": top3[2]
    })


In [145]:
df_recomendacoes = pd.DataFrame(lista_recomendacoes)

print(df_recomendacoes.head())

   member_id  cluster_label recommended_game_1 recommended_game_2  \
0        1.0            2.0              G0002              G0011   
1        2.0            1.0              G0036              G0006   
2        3.0            1.0              G0055              G0036   
3        4.0            1.0              G0055              G0036   
4        5.0            2.0              G0020              G0011   

  recommended_game_3  
0              G0075  
1              G0037  
2              G0074  
3              G0006  
4              G0064  


In [146]:
print(df_recomendacoes.head())

cliente_teste_id = df_recomendacoes.iloc[0]['member_id']
cluster_teste = df_recomendacoes.iloc[0]['cluster_label']

recs_teste = [
    df_recomendacoes.iloc[0]['recommended_game_1'],
    df_recomendacoes.iloc[0]['recommended_game_2'],
    df_recomendacoes.iloc[0]['recommended_game_3']

]

   member_id  cluster_label recommended_game_1 recommended_game_2  \
0        1.0            2.0              G0002              G0011   
1        2.0            1.0              G0036              G0006   
2        3.0            1.0              G0055              G0036   
3        4.0            1.0              G0055              G0036   
4        5.0            2.0              G0020              G0011   

  recommended_game_3  
0              G0075  
1              G0037  
2              G0074  
3              G0006  
4              G0064  


In [147]:
compras_reais = df_orders[df_orders['member_id'] == cliente_teste_id]['game_id'].tolist()

print(f'\n --- Teste de integridade cliente {cliente_teste_id} do cluster {cluster_teste} ---')
print(f'\n --- jogos recomendados: {recs_teste} ---')
print(f'\n --- jogos que ele ja acessou: {compras_reais[:5]}... (total{len(compras_reais)}) ---')


 --- Teste de integridade cliente 1.0 do cluster 2.0 ---

 --- jogos recomendados: ['G0002', 'G0011', 'G0075'] ---

 --- jogos que ele ja acessou: ['G0007', 'G0020', 'G0040', 'G0074', 'G0057']... (total10) ---


In [148]:
intersecao = set(recs_teste).intersection(set(compras_reais))
if len(intersecao) ==0:
    print("SUCESSO: NENHUM JOGO RECOMENDADO FOI COMPRADO ANTERIORMENTE")
else:
    print(f"X ERROR CRITICO: O SISTEMA RECOMENDOU JOGOS JA COMPRADOS{intersecao}")


SUCESSO: NENHUM JOGO RECOMENDADO FOI COMPRADO ANTERIORMENTE


In [149]:
print("Corrigindo tipos de dados para o padrao exigido")
df_recomendacoes['member_id'] = df_recomendacoes['member_id'].astype(int)
df_recomendacoes['cluster_label'] = df_recomendacoes['cluster_label'].astype(int)

print("\n --- Estrutura final (dtypes) do df de recomendações")
print(df_recomendacoes.dtypes)

print("\n ----- Primeiras 5 linhas -----")
print(df_recomendacoes.head())

Corrigindo tipos de dados para o padrao exigido

 --- Estrutura final (dtypes) do df de recomendações
member_id             int64
cluster_label         int64
recommended_game_1      str
recommended_game_2      str
recommended_game_3      str
dtype: object

 ----- Primeiras 5 linhas -----
   member_id  cluster_label recommended_game_1 recommended_game_2  \
0          1              2              G0002              G0011   
1          2              1              G0036              G0006   
2          3              1              G0055              G0036   
3          4              1              G0055              G0036   
4          5              2              G0020              G0011   

  recommended_game_3  
0              G0075  
1              G0037  
2              G0074  
3              G0006  
4              G0064  


In [150]:
caminho_csv = r"C:\GameStoreBrasil\output\Session1_Segmentation_Recomendations_GameStore.csv"

df_recomendacoes.to_csv(caminho_csv, index = False)